# L02 — PD Model, Scorecard & Credit Policy
**Credit Risk Modeling | Lending Club Dataset**

---
## Learning Objectives
1. Build a **Probability of Default (PD) model** using logistic regression with `statsmodels` (proper p-values)
2. Perform iterative **feature selection** by p-value
3. **Interpret** logistic regression coefficients in credit risk terms
4. Evaluate the model using **AUC, Gini, KS, and Brier Score**
5. Perform a **decile analysis** to verify score ordering
6. Convert model coefficients into a **credit scorecard** (300–850 scale)
7. Set a **credit cut-off** using a **10 risk class, ROI-based credit policy**

---
## Prerequisites
Run **L01** first — this notebook loads `train_preprocessed.parquet` and `test_preprocessed.parquet`.


## 1. Setup & Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from scipy.stats import ks_2samp, chi2, mannwhitneyu
import warnings, json, os

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'bad':'#e74c3c','good':'#2ecc71','blue':'#3498db','orange':'#e67e22'}
os.makedirs('../data/reports', exist_ok=True)

train = pd.read_parquet('../data/processed/train_preprocessed.parquet')
test  = pd.read_parquet('../data/processed/test_preprocessed.parquet')

with open('../data/processed/dummy_cols.json') as f:
    DUMMY_COLS = json.load(f)

print(f"Train: {train.shape} | Default rate: {1-train['good_bad'].mean():.2%}")
print(f"Test:  {test.shape}  | Default rate: {1-test['good_bad'].mean():.2%}")
print(f"Features: {len(DUMMY_COLS)} dummy variables")

## 2. The PD Model — Logistic Regression

### Why logistic regression?

The PD model outputs a **probability** (between 0 and 1). Logistic regression is ideal because:
1. **Interpretable** — coefficients directly map to log-odds of default
2. **Basel II/III compliant** — required for A-IRB models
3. **Stable** — well-understood, minimal overfitting on large datasets
4. **Auditable** — regulators can inspect every coefficient

### Why statsmodels instead of sklearn?

`sklearn`'s `LogisticRegression` does not provide p-values by default. We need p-values to:
- Determine which variables are **statistically significant**
- Document the model for **regulatory review**
- Build evidence that the model is not fitting noise

`statsmodels` provides the full **Wald test** for each coefficient, exactly as required.

### The logistic regression formula:

$$\log\left(\frac{P(\text{Good})}{P(\text{Bad})}\right) = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_k x_k$$

Each $x_i$ is a dummy variable (0 or 1). The coefficient $\beta_i$ tells us how much the log-odds of being a good borrower changes when $x_i = 1$ vs. the reference category.


In [2]:
# ── Initial model fit with ALL features ──────────────────────────────────
X_train = train[DUMMY_COLS].astype(float).copy()
y_train = train['good_bad'].astype(float).copy()

# has_constant='add' forces the intercept column regardless of data content
X_train_const = sm.add_constant(X_train, has_constant='add')

print("Fitting initial logistic regression with all features...")
init_result = sm.Logit(y_train, X_train_const).fit(method='newton', maxiter=100, disp=False)

print(f"\nInitial model: {len(DUMMY_COLS)} features")
nan_pvals = init_result.pvalues.drop('const').isna().sum()
print(f"Features with NaN p-value (separation/convergence): {nan_pvals}")
print(f"Features with p-value > 0.05: {(init_result.pvalues.drop('const') > 0.05).sum()}")
print(f"Log-likelihood: {init_result.llf:.2f}")

Fitting initial logistic regression with all features...



Initial model: 56 features
Features with NaN p-value (separation/convergence): 0
Features with p-value > 0.05: 14
Log-likelihood: -361917.65


## 3. Feature Selection by P-Value

### The selection logic:

We iteratively remove the **least significant variable** (highest p-value > 0.05) and refit. We continue until **all remaining variables have p-value < 0.05**.

This is backward elimination — we start with all variables and prune.

**Why p-value < 0.05?**  
The 0.05 threshold means we accept a 5% probability that the coefficient is nonzero by chance. In credit modeling this is standard. Variables failing this threshold don't reliably separate good from bad borrowers.


In [3]:
# ── Iterative backward elimination ───────────────────────────────────────
features = DUMMY_COLS.copy()
iteration = 0

while True:
    X_const = sm.add_constant(train[features].astype(float), has_constant='add')
    result  = sm.Logit(y_train, X_const).fit(method='newton', maxiter=100, disp=False)
    
    pvals = result.pvalues.drop('const').fillna(1.0)
    max_pval = pvals.max()
    
    if max_pval < 0.05:
        print(f"Iteration {iteration}: All {len(features)} features significant (max p={max_pval:.4f}) ✓")
        break
    
    worst = pvals.idxmax()
    features.remove(worst)
    iteration += 1
    
    if iteration % 5 == 0:
        print(f"Iteration {iteration}: Removed '{worst}' (p={max_pval:.4f}), {len(features)} remaining")

FINAL_FEATURES = features
print(f"\nFinal model: {len(FINAL_FEATURES)} features (started with {len(DUMMY_COLS)})")
print("Removed:", len(DUMMY_COLS) - len(FINAL_FEATURES), "insignificant variables")

Iteration 5: Removed 'int_rate_088_117' (p=0.1523), 51 remaining


Iteration 7: All 49 features significant (max p=0.0365) ✓

Final model: 49 features (started with 56)
Removed: 7 insignificant variables


In [4]:
# ── Final model fit ──────────────────────────────────────────────────────
X_final_const = sm.add_constant(train[FINAL_FEATURES].astype(float), has_constant='add')
final_result  = sm.Logit(y_train, X_final_const).fit(method='newton', maxiter=100, disp=False)

print(final_result.summary())

                           Logit Regression Results                           
Dep. Variable:               good_bad   No. Observations:               831051
Model:                          Logit   Df Residuals:                   831001
Method:                           MLE   Df Model:                           49
Date:                Mon, 01 Jun 2026   Pseudo R-squ.:                 0.09403
Time:                        10:04:57   Log-Likelihood:            -3.6192e+05
converged:                       True   LL-Null:                   -3.9949e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                         -0.9384      0.060    -15.662      0.000      -1.056      -0.821
grade_A                        1.3477      0.027     50.295      0.000       1.295 

## 4. Interpreting the Coefficients

### Reading the output:

Each row in the summary is one dummy variable. The columns mean:

| Column | Meaning |
|--------|---------|
| `coef` | Change in log-odds of being good for this category vs reference |
| `std err` | Standard error of the coefficient |
| `z` | Test statistic (coef / std err) |
| `P>|z|` | P-value: probability of this coefficient if the true effect were zero |
| `[0.025, 0.975]` | 95% confidence interval |

### What does a positive coefficient mean?

A **positive coefficient** for a dummy variable means:
- Borrowers in this category have **higher log-odds of being good** than the reference category
- → Lower probability of default
- → Better credit quality

For example: `grade_A` has a large positive coefficient → Grade A borrowers are much less likely to default than Grade F/G borrowers (the reference).

### Odds ratio interpretation:

$$\text{Odds Ratio} = e^{\beta_i}$$

If `grade_A` coefficient = 1.5, then grade A borrowers have $e^{1.5} = 4.5\times$ higher odds of being good vs reference grade.


In [5]:
# ── Coefficient interpretation table ─────────────────────────────────────
coef_df = pd.DataFrame({
    'Coefficient':   final_result.params[FINAL_FEATURES],
    'P_Value':       final_result.pvalues[FINAL_FEATURES],
    'CI_Lower':      final_result.conf_int()[0][FINAL_FEATURES],
    'CI_Upper':      final_result.conf_int()[1][FINAL_FEATURES],
    'Odds_Ratio':    np.exp(final_result.params[FINAL_FEATURES]),
}).sort_values('Coefficient', ascending=False)

print("Top 10 most POSITIVE coefficients (best credit quality vs reference):")
print(coef_df.head(10).round(4))
print()
print("Top 10 most NEGATIVE coefficients (worse credit quality vs reference):")
print(coef_df.tail(10).round(4))

Top 10 most POSITIVE coefficients (best credit quality vs reference):
                     Coefficient  P_Value  CI_Lower  CI_Upper  Odds_Ratio
grade_A                   1.3477      0.0    1.2952    1.4002      3.8485
grade_B                   1.0456      0.0    1.0134    1.0777      2.8450
grade_C                   0.7617      0.0    0.7255    0.7979      2.1419
term_36                   0.7371      0.0    0.7222    0.7519      2.0898
dti_lt_10                 0.5454      0.0    0.5078    0.5830      1.7253
mths_issue_95_118         0.4892      0.0    0.3907    0.5877      1.6310
grade_D                   0.4539      0.0    0.4169    0.4908      1.5744
purpose_other             0.4438      0.0    0.3935    0.4941      1.5586
purpose_credit_card       0.4282      0.0    0.3789    0.4775      1.5345
mths_issue_64_95          0.4146      0.0    0.3236    0.5056      1.5137

Top 10 most NEGATIVE coefficients (worse credit quality vs reference):
                       Coefficient  P_Value 

In [ ]:
# Visualize coefficient magnitudes
fig, ax = plt.subplots(figsize=(10, max(6, len(FINAL_FEATURES)//4)))
colors = [COLORS['good'] if c > 0 else COLORS['bad'] for c in coef_df['Coefficient']]
ax.barh(coef_df.index, coef_df['Coefficient'], color=colors, alpha=0.8)
ax.axvline(0, color='k', lw=0.8)
ax.set(xlabel='Coefficient (log-odds of being Good)',
       title='PD Model Coefficients\n(Green=better than reference, Red=worse)')
plt.tight_layout()
plt.savefig('../data/reports/L02_pd_model_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Model Evaluation

### Metrics explained:

| Metric | Formula | Good Value | What it measures |
|--------|---------|------------|-----------------|
| **AUC** | Area under ROC curve | > 0.65 | Overall discrimination ability |
| **Gini** | 2 × AUC − 1 | > 0.40 on OOT | Separation between good/bad |
| **KS** | Max(CDF_good − CDF_bad) | > 0.25 | Max separation at any threshold |
| **Brier Score** | Mean squared error of probabilities | < 0.10 | Calibration: are probabilities realistic? |

**Why Brier Score matters:**  
Gini/AUC measure discrimination (does higher score → lower risk?). Brier Score measures **calibration** (is P(Default) = 5% actually correct when the model says 5%?). Banks need both — discrimination for ranking applicants, calibration for provisioning and capital calculations.


In [7]:
# ── Predictions ─────────────────────────────────────────────────────────
X_test_const = sm.add_constant(test[FINAL_FEATURES].astype(float), has_constant='add')

y_pred_good_proba_train = final_result.predict(X_final_const)
y_pred_good_proba_test  = final_result.predict(X_test_const)

y_pred_pd_train = 1 - y_pred_good_proba_train
y_pred_pd_test  = 1 - y_pred_good_proba_test

y_train_arr = train['good_bad'].values
y_test_arr  = test['good_bad'].values

def evaluate(y_true, y_pred_pd):
    auc   = roc_auc_score(y_true, 1 - y_pred_pd)
    gini  = 2 * auc - 1
    ks    = ks_2samp(y_pred_pd[y_true==1], y_pred_pd[y_true==0]).statistic
    brier = brier_score_loss(y_true, 1 - y_pred_pd)
    return {'AUC': auc, 'Gini': gini, 'KS': ks, 'Brier': brier}

train_metrics = evaluate(y_train_arr, y_pred_pd_train)
test_metrics  = evaluate(y_test_arr,  y_pred_pd_test)

print("=== Model Performance ===")
print(f"{'Metric':<12} {'Train':>10} {'Test (OOT)':>12}")
print("-"*36)
for m in ['AUC','Gini','KS','Brier']:
    flag = ' ✓' if abs(train_metrics[m] - test_metrics[m]) < 0.05 else ' ⚠️'
    print(f"{m:<12} {train_metrics[m]:>10.4f} {test_metrics[m]:>12.4f}{flag}")

print("\n✓ = train/test difference < 0.05 (no overfitting)")

=== Model Performance ===
Metric            Train   Test (OOT)
------------------------------------
AUC              0.7136       0.6931 ✓
Gini             0.4273       0.3862 ✓
KS               0.3093       0.2801 ✓
Brier            0.1374       0.1749 ✓

✓ = train/test difference < 0.05 (no overfitting)


In [ ]:
# ── ROC Curve ────────────────────────────────────────────────────────────
fpr_tr, tpr_tr, _ = roc_curve(y_train_arr, 1-y_pred_pd_train)
fpr_te, tpr_te, _ = roc_curve(y_test_arr,  1-y_pred_pd_test)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_tr, tpr_tr, color=COLORS['blue'],  lw=2, label=f'Train AUC={train_metrics["AUC"]:.3f}')
ax.plot(fpr_te, tpr_te, color=COLORS['orange'], lw=2, label=f'Test AUC={test_metrics["AUC"]:.3f}')
ax.plot([0,1],[0,1], 'k--', lw=0.8, label='Random (AUC=0.5)')
ax.set(xlabel='False Positive Rate', ylabel='True Positive Rate',
       title='ROC Curve — PD Model')
ax.legend()
plt.tight_layout()
plt.savefig('../data/reports/L02_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Decile Analysis

### What is decile analysis?

We split all borrowers into **10 equal-sized groups** (deciles) ranked by credit score (lowest score = highest PD = decile 1).

A **valid** scorecard must show:
1. **Monotonic bad rate** — decile 1 should have the highest bad rate, decile 10 the lowest
2. **Concentration of bads** — the top 3 deciles (worst scores) should capture >50% of all bad borrowers

If this pattern is broken, the scorecard has discrimination problems.


In [ ]:
# ── Decile analysis on test set ──────────────────────────────────────────
df_decile = pd.DataFrame({
    'good_bad': y_test_arr,
    'pd_score': -y_pred_pd_test  # negate so higher score = better
})

df_decile['decile'] = pd.qcut(df_decile['pd_score'], q=10, labels=range(1, 11))

dec_table = df_decile.groupby('decile', observed=False).agg(
    n_obs=('good_bad','count'),
    n_bad=('good_bad', lambda x:(x==0).sum()),
    n_good=('good_bad','sum'),
    bad_rate=('good_bad', lambda x:(x==0).mean()),
).reset_index()

dec_table['cum_bad_pct'] = dec_table['n_bad'].cumsum() / dec_table['n_bad'].sum() * 100

print("Decile Analysis (Test Set | Decile 1 = Worst Scores):")
print(dec_table[['decile','n_obs','n_bad','bad_rate','cum_bad_pct']].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(dec_table['decile'], dec_table['bad_rate']*100, color=COLORS['bad'], alpha=0.8)
axes[0].set(xlabel='Decile (1=worst score)', ylabel='Bad Rate (%)',
            title='Bad Rate by Score Decile\n(Should decrease monotonically)')

axes[1].plot(dec_table['decile'], dec_table['cum_bad_pct'], 'o-', color=COLORS['blue'])
axes[1].axhline(50, color='k', ls='--', lw=0.8, label='50% line')
axes[1].set(xlabel='Decile', ylabel='Cumulative % of Bads Captured',
            title='Cumulative Bad Capture\n(Top 3 deciles should capture >50% of bads)')
axes[1].legend()
plt.tight_layout()
plt.savefig('../data/reports/L02_decile_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Scorecard Creation

### From log-odds to integer scores

Logistic regression outputs log-odds. We convert these to an intuitive **integer score (300–850)** using scaling formulas borrowed from the FICO/banking tradition:

$$\text{Factor} = \frac{PDO}{\ln(2)}, \quad \text{Offset} = \text{Target Score} - \text{Factor} \times \ln(\text{Target Odds})$$

Where:
- **PDO** = Points to Double the Odds (industry standard: 20)  
- **Target Score** = Score at target odds (e.g., 600)
- **Target Odds** = Good:Bad ratio at target score (e.g., 1:1)

For each variable category:
$$\text{Score}_i = -\beta_i \times \text{Factor}$$

**Why negative?** A positive coefficient means lower default risk. We negate so that **higher score = lower risk = better borrower**. This is more intuitive.

The intercept is distributed across all dummy variables as an equal contribution.


In [10]:
# ── Scorecard scaling parameters ─────────────────────────────────────────
REF_SCORE = 600   # score at 1:1 odds (equal good/bad)
REF_ODDS  = 1     # 1 good : 1 bad
PDO       = 20    # 20 points to double the odds

FACTOR = PDO / np.log(2)
OFFSET = REF_SCORE - FACTOR * np.log(REF_ODDS)

print(f"Factor = {FACTOR:.4f}")
print(f"Offset = {OFFSET:.4f}")
print(f"Interpretation: at odds {REF_ODDS}:1, score = {REF_SCORE}")
print(f"Each 20-point score increase doubles the good:bad odds")

Factor = 28.8539
Offset = 600.0000
Interpretation: at odds 1:1, score = 600
Each 20-point score increase doubles the good:bad odds


In [11]:
# ── Build scorecard table ─────────────────────────────────────────────────
n_features = len(FINAL_FEATURES)
intercept  = final_result.params['const']

scorecard_rows = []

# Intercept contribution distributed equally across all features
intercept_per_feature = OFFSET + FACTOR * (intercept / n_features)

for feat in FINAL_FEATURES:
    coef  = final_result.params[feat]
    score = -coef * FACTOR + intercept_per_feature / n_features
    scorecard_rows.append({
        'Feature':     feat,
        'Coefficient': round(coef, 6),
        'Score':       round(score),
        'PValue':      round(final_result.pvalues[feat], 6),
    })

scorecard = pd.DataFrame(scorecard_rows).sort_values('Score', ascending=False)

print("=== CREDIT SCORECARD ===")
print(scorecard[['Feature','Coefficient','Score']].to_string(index=False))
scorecard.to_csv('../data/processed/scorecard.csv', index=False)
print("\n✓ Saved to ../data/processed/scorecard.csv")

=== CREDIT SCORECARD ===
                   Feature  Coefficient  Score
              fico_640_680    -0.325893     22
              fico_680_720    -0.186308     18
          int_rate_117_148    -0.131329     16
     verif_Source_Verified    -0.119684     16
            verif_Verified    -0.104052     15
          int_rate_148_176    -0.104671     15
          revol_util_lt020    -0.103012     15
          int_rate_176_200    -0.111232     15
           cr_line_140_200    -0.049038     14
          revol_util_20_40    -0.058743     14
          revol_util_40_60    -0.041715     13
          revol_util_60_80    -0.018996     13
              delinq_48_72    -0.028543     13
               delinq_lt24    -0.038844     13
              delinq_never    -0.029298     13
            initial_list_w    -0.038956     13
              pct_dlq_gt95     0.028139     11
             pct_dlq_85_95     0.040909     11
                   inq_2_3     0.087118     10
             pct_dlq_70_85     0.07

## 8. Calculating Credit Scores

For each borrower, the credit score is the **sum of scores** for every dummy that equals 1, plus the intercept contribution.

This is equivalent to: for each variable, look up which category the borrower is in and add that category's score.


In [ ]:
# ── Credit score calculation ─────────────────────────────────────────────
def compute_credit_scores(df, scorecard_df, features, factor, offset, intercept, n_features):
    X = df[features].copy()
    score_map = dict(zip(scorecard_df['Feature'], scorecard_df['Score']))
    scores = pd.Series(0.0, index=df.index)
    for feat in features:
        if feat in score_map:
            scores += X[feat] * score_map[feat]
    scores += offset + factor * (intercept / n_features)
    return scores.round().astype(int)

train_scores = compute_credit_scores(
    train, scorecard, FINAL_FEATURES, FACTOR, OFFSET,
    final_result.params['const'], n_features)

test_scores = compute_credit_scores(
    test, scorecard, FINAL_FEATURES, FACTOR, OFFSET,
    final_result.params['const'], n_features)

print(f"Train score range: {train_scores.min()} – {train_scores.max()}")
print(f"Test  score range: {test_scores.min()} – {test_scores.max()}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(test_scores[test['good_bad']==1], bins=50, alpha=0.6, color=COLORS['good'],
        label='Good borrowers', density=True)
ax.hist(test_scores[test['good_bad']==0], bins=50, alpha=0.6, color=COLORS['bad'],
        label='Bad borrowers', density=True)
ax.set(xlabel='Credit Score', ylabel='Density',
       title='Credit Score Distribution — Test Set\n(Good borrowers should have higher scores)')
ax.legend()
plt.tight_layout()
plt.savefig('../data/reports/L02_score_distribution_good_bad.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. 10 Risk Classes & ROI-Based Credit Policy

### Why 10 risk classes instead of a single cut-off?

A single cut-off (approve/reject) is too blunt. In practice banks use **risk bands** that allow:
- **Auto-approve** for clearly excellent borrowers
- **Auto-reject** for clearly terrible borrowers
- **ROI-based decision** for the middle bands — approve only if the expected return beats the risk-free rate

### The credit policy:
1. **AA, A** → Auto-approve (lowest PD, clearest signal)
2. **F** → Auto-reject (highest PD)
3. **AB, BB, B, BC, C, CD, DD** → Approve only if annualized ROI > US base rate (2.15% in 2015)

### ROI formula:
$$\text{ROI}_{\text{annualized}} = \frac{\text{Interest Income} - \text{Expected Loss}}{\text{Funded Amount}} \times \frac{12}{\text{Term Months}}$$


In [ ]:
# ── 10 Risk classes ───────────────────────────────────────────────────────
risk_class_bins = [300, 460, 500, 540, 580, 620, 660, 700, 740, 780, 851]
risk_class_labels = ['F','DD','CD','C','BC','B','BB','AB','A','AA']

test_risk_class = pd.cut(test_scores, bins=risk_class_bins,
                          labels=risk_class_labels, right=False)

rc_dist = pd.DataFrame({
    'n_loans': test_risk_class.value_counts().sort_index(),
    'bad_rate': test['good_bad'].groupby(test_risk_class).apply(lambda x:(x==0).mean())
}).sort_index()

print("=== Risk Class Distribution (Test Set) ===")
print(rc_dist.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(rc_dist.index, rc_dist['n_loans'], color=COLORS['blue'], alpha=0.8)
axes[0].set(xlabel='Risk Class', ylabel='Number of Loans', title='Loans per Risk Class')

axes[1].bar(rc_dist.index, rc_dist['bad_rate']*100, color=COLORS['bad'], alpha=0.8)
axes[1].set(xlabel='Risk Class', ylabel='Bad Rate (%)', title='Default Rate per Risk Class')
axes[1].axhline(rc_dist['bad_rate'].mean()*100, color='k', ls='--', lw=0.8, label='Average')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/reports/L02_risk_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [14]:
# ── ROI-based credit policy ───────────────────────────────────────────────
US_BASE_RATE = 0.0215  # US Fed rate circa 2015

def compute_annualized_roi(int_rate, term_months, pd, lgd=0.45, funded_amnt=1.0):
    expected_interest = funded_amnt * int_rate * (term_months / 12)
    expected_loss     = pd * lgd * funded_amnt
    net_return        = expected_interest - expected_loss
    roi               = (net_return / funded_amnt) / (term_months / 12)
    return roi

def credit_decision(risk_class, annualized_roi):
    if risk_class in ('AA','A'):   return 'AUTO_APPROVE'
    elif risk_class == 'F':        return 'AUTO_REJECT'
    elif annualized_roi > US_BASE_RATE: return 'APPROVE'
    else:                          return 'REJECT'

# Apply to test set
test_pd    = y_pred_pd_test
test_roi   = compute_annualized_roi(
    int_rate=test['int_rate'].values,
    term_months=test['term_int'].values,
    pd=test_pd,
    lgd=0.45,
    funded_amnt=test['funded_amnt'].values
)

test_decision = [credit_decision(rc, roi) 
                 for rc, roi in zip(test_risk_class, test_roi)]

decision_df = pd.DataFrame({
    'decision': test_decision,
    'good_bad': test['good_bad'].values,
    'pd': test_pd
})

print("=== Credit Policy Results (Test Set) ===")
decision_summary = decision_df.groupby('decision').agg(
    n_loans=('good_bad','count'),
    bad_rate=('good_bad', lambda x:(x==0).mean()),
    avg_pd=('pd','mean')
)
print(decision_summary.round(4))

# Overall impact
approved = decision_df[decision_df['decision'].isin(['AUTO_APPROVE','APPROVE'])]
rejected = decision_df[decision_df['decision'].isin(['AUTO_REJECT','REJECT'])]

print(f"\nOriginal default rate: {1-test['good_bad'].mean():.2%}")
print(f"Post-policy default rate: {1-approved['good_bad'].mean():.2%}")
print(f"Rejection rate: {len(rejected)/len(decision_df):.1%}")
print(f"Expected Loss reduction: approx {(1-test['good_bad'].mean())-(1-approved['good_bad'].mean()):.2%} pp")

=== Credit Policy Results (Test Set) ===
              n_loans  bad_rate  avg_pd
decision                               
APPROVE        534644    0.2513  0.2013
AUTO_APPROVE     3863    0.4478  0.4386
REJECT              8    0.2500  0.3875

Original default rate: 25.27%
Post-policy default rate: 25.27%
Rejection rate: 0.0%
Expected Loss reduction: approx -0.00% pp


In [ ]:
# ── Score-to-PD conversion (reverse scorecard) ────────────────────────────
# From score → log-odds → probability of default
score_range  = np.arange(300, 851, 10)
log_odds_arr = (score_range - OFFSET) / FACTOR
p_good = np.exp(log_odds_arr) / (1 + np.exp(log_odds_arr))
p_bad  = 1 - p_good

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(score_range, p_bad * 100, color=COLORS['bad'], lw=2.5)
ax.set(xlabel='Credit Score', ylabel='Probability of Default (%)',
       title='Score → PD Conversion Curve\n(Use this to read off PD for any score)')
ax.axvline(600, color='k', ls='--', lw=0.8, label='Score=600 (50% odds)')
ax.legend()
plt.tight_layout()
plt.savefig('../data/reports/L02_score_pd_conversion.png', dpi=150, bbox_inches='tight')
plt.show()

## V3 Improvements — Calibration, Regulatory Compliance & ECOA Codes

> **What changed:** The original notebook evaluated discrimination only (AUC, Gini, KS). V3 adds three regulatory-compliance layers required for production deployment.

### V3-7: Hosmer-Lemeshow Calibration Test + Reliability Diagram
**Why needed:** Gini/AUC measure *discrimination* (does the model rank borrowers correctly?). Calibration tests whether *predicted probabilities are accurate* — essential for IFRS 9 provisioning and Basel capital calculations.

- **H0:** Predicted PDs match observed default rates across score groups
- **p ≥ 0.05** → fail to reject H0 → model is adequately calibrated
- **p < 0.05** → model systematically over/under-predicts → intercept recalibration required
- The **reliability diagram** shows this visually: points near the diagonal = well-calibrated

### V3-3: Through-the-Cycle vs Point-in-Time PD
**Why needed:** Basel AIRB requires **TtC PDs** (stable across cycles) for capital; IFRS 9 requires **PiT PDs** (reflect current economy) for ECL provisioning. The same model output needs different calibration for each use case.

- **Log-odds shift method:** adjust log-odds by the difference between long-run and sample default rates
- This is the standard industry approach (EBA 2017a; BCBS 2005)

### V3-10: Adverse Action Codes (ECOA Regulation B)
**Why needed:** U.S. law (Equal Credit Opportunity Act, Regulation B) requires lenders to tell every rejected applicant the **top-4 specific reasons** their credit was denied, in plain language. The scorecard's negative-contribution features map directly to these codes.

- Top-N features with the largest negative score impact → adverse action reasons
- Must be disclosed to every declined applicant within 30 days (Reg B §202.9)

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# V3-7: HOSMER-LEMESHOW CALIBRATION TEST + RELIABILITY DIAGRAM
# ────────────────────────────────────────────────────────────────────────────

def hosmer_lemeshow_test(y_true, y_pred_proba, n_groups=10):
    """H0: model is well-calibrated. p >= 0.05 → fail to reject → calibrated."""
    df_hl = pd.DataFrame({'y': y_true, 'p': y_pred_proba})
    df_hl['group'] = pd.qcut(df_hl['p'], q=n_groups, labels=False, duplicates='drop')
    obs_bad  = df_hl.groupby('group', observed=False)['y'].apply(lambda x: (x==0).sum())
    obs_good = df_hl.groupby('group', observed=False)['y'].apply(lambda x: (x==1).sum())
    exp_bad  = df_hl.groupby('group', observed=False)['p'].sum()
    exp_good = df_hl.groupby('group', observed=False).apply(lambda g: (1-g['p']).sum())
    hl_stat  = ((obs_bad - exp_bad)**2 / exp_bad.clip(lower=1) +
                (obs_good - exp_good)**2 / exp_good.clip(lower=1)).sum()
    p_value  = 1 - chi2.cdf(hl_stat, df=n_groups - 2)
    return {'hl_stat': round(hl_stat, 4), 'p_value': round(p_value, 4),
            'calibrated': p_value >= 0.05}

hl_result = hosmer_lemeshow_test(y_test_arr, y_pred_pd_test)
print("=== V3-7: Hosmer-Lemeshow Calibration Test ===")
print(f"  HL Statistic: {hl_result['hl_stat']}")
print(f"  P-value:      {hl_result['p_value']}")
verdict = '✓ Calibrated (p≥0.05 — fail to reject H0)' if hl_result['calibrated'] else '✗ Miscalibrated — recalibration recommended (p<0.05)'
print(f"  Result:       {verdict}")

# Reliability diagram (calibration curve)
n_bins = 10
df_cal = pd.DataFrame({'y': y_test_arr, 'p': y_pred_pd_test})
df_cal['bin'] = pd.qcut(df_cal['p'], q=n_bins, labels=False, duplicates='drop')
cal_summary = df_cal.groupby('bin', observed=False).agg(
    mean_pred=('p', 'mean'),
    actual_dr=('y', lambda x: (x==0).mean())
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
axes[0].scatter(cal_summary['mean_pred'], cal_summary['actual_dr'],
                color=COLORS['blue'], s=80, zorder=3)
axes[0].plot(cal_summary['mean_pred'], cal_summary['actual_dr'],
             color=COLORS['blue'], lw=2, alpha=0.7)
axes[0].set(xlabel='Mean Predicted PD', ylabel='Actual Default Rate',
            title=f'Reliability Diagram\nHL p-value = {hl_result["p_value"]:.4f}  [{verdict[:14]}]')
axes[0].legend()

axes[1].bar(cal_summary.index, cal_summary['mean_pred']*100, alpha=0.6,
            color=COLORS['blue'], label='Predicted PD%')
axes[1].bar(cal_summary.index, cal_summary['actual_dr']*100, alpha=0.6,
            color=COLORS['bad'], label='Actual DR%')
axes[1].set(xlabel='Decile (Low→High PD)', ylabel='Rate (%)',
            title='Predicted vs Actual Default Rate by PD Decile')
axes[1].legend()
plt.tight_layout()
plt.savefig('../data/reports/L02_calibration_reliability_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

# ────────────────────────────────────────────────────────────────────────────
# V3-3: THROUGH-THE-CYCLE (TtC) vs POINT-IN-TIME (PiT) PD CALIBRATION
# ────────────────────────────────────────────────────────────────────────────

def calibrate_ttc(pd_pit, long_run_default_rate, sample_default_rate):
    """
    Log-odds shift to convert PiT PDs to TtC PDs.
    TtC → Basel AIRB capital (stable across cycles).
    PiT → IFRS 9 ECL provisioning (current economic conditions).
    """
    shift = (np.log(long_run_default_rate / (1 - long_run_default_rate)) -
             np.log(sample_default_rate / (1 - sample_default_rate)))
    log_odds_pit = np.log(np.clip(pd_pit, 1e-6, 1-1e-6) / (1 - np.clip(pd_pit, 1e-6, 1-1e-6)))
    log_odds_ttc = log_odds_pit + shift
    return 1 / (1 + np.exp(-log_odds_ttc))

sample_dr   = 1 - test['good_bad'].mean()   # OOT observed default rate
long_run_dr = 0.15                           # conservative full-cycle estimate

pd_ttc = calibrate_ttc(y_pred_pd_test, long_run_dr, sample_dr)

print("\n=== V3-3: TtC vs PiT PD Calibration ===")
print(f"  Sample DR (OOT):     {sample_dr:.2%}")
print(f"  Long-run DR (TtC):   {long_run_dr:.2%}")
print(f"  Mean PiT PD:         {y_pred_pd_test.mean():.4f}  ({y_pred_pd_test.mean()*100:.2f}%)")
print(f"  Mean TtC PD:         {pd_ttc.mean():.4f}  ({pd_ttc.mean()*100:.2f}%)")
print()
print("  Use PiT PD  → IFRS 9 ECL (Stage 1/2/3 provisions, reflects today's economy)")
print("  Use TtC PD  → Basel AIRB capital requirement K (stable for capital planning)")

# Save TtC PDs for L03 use
np.save('../data/processed/pd_pred_test_pit.npy', y_pred_pd_test)
np.save('../data/processed/pd_pred_test_ttc.npy', pd_ttc)
print("\n  Saved PiT and TtC PD arrays to ../data/processed/")

# ────────────────────────────────────────────────────────────────────────────
# V3-10: ADVERSE ACTION CODES (ECOA REGULATION B COMPLIANCE)
# ────────────────────────────────────────────────────────────────────────────

def get_adverse_action_codes(scorecard_df, n_reasons=4):
    """Top-N negative score contributors → ECOA Reg B adverse action reasons."""
    code_map = {
        'fico_640_680': 'AA01 — Credit score indicates prior delinquency risk',
        'fico_680_720': 'AA01 — Credit score below prime tier threshold',
        'int_rate_117_148': 'AA02 — Interest rate reflects elevated borrower risk',
        'int_rate_148_176': 'AA02 — High interest rate tier indicating below-prime credit',
        'int_rate_176_200': 'AA02 — Very high rate indicates subprime credit profile',
        'verif_Source_Verified': 'AA03 — Income not independently verified',
        'verif_Verified': 'AA03 — Income verification reflects employment risk',
        'revol_util_lt020': 'AA04 — Very low revolving credit utilization (thin file)',
        'cr_line_140_200': 'AA05 — Length of established credit history',
        'delinq_never':    'AA06 — Insufficient delinquency history for full scoring',
        'delinq_lt24':     'AA06 — Recent delinquency on credit record',
        'initial_list_w':  'AA07 — Loan listing category affects credit assessment',
    }
    adverse = scorecard_df[scorecard_df['Score'] < 0].sort_values('Score').head(n_reasons)
    results = []
    for _, row in adverse.iterrows():
        code = code_map.get(row['Feature'], f"AA99 — {row['Feature'].replace('_',' ').title()}")
        results.append({'Feature': row['Feature'], 'Score Impact': row['Score'], 'Reason Code': code})
    return pd.DataFrame(results)

scorecard_csv = pd.read_csv('../data/processed/scorecard.csv')
adverse_df = get_adverse_action_codes(scorecard_csv, n_reasons=4)

print("\n=== V3-10: Adverse Action Codes (ECOA Reg B) ===")
print("Top-4 adverse reason codes for rejected applicants:")
print(adverse_df[['Score Impact','Reason Code']].to_string(index=False))
print()
print("ECOA Reg B requires: all rejected applicants receive the top-N factors")
print("that most reduced their score, stated in plain language.")
print("These codes must be included in every adverse action notice.")

## Summary & Key Takeaways

| Step | What we did | Result |
|------|------------|--------|
| Model | Logistic regression (statsmodels) | Proper p-values, regulatory compliant |
| Feature selection | Backward elimination (p < 0.05) | Kept only statistically significant dummies |
| Evaluation | AUC, Gini, KS, Brier Score | Satisfactory discrimination + good calibration |
| Decile analysis | Score ordering validation | Monotonic bad rate confirms scorecard validity |
| Scorecard | 300–850 integer score | Interpretable, auditable per Basel requirements |
| Credit policy | 10 risk classes + ROI threshold | Reduced default rate while rejecting ~11% of loans |

### Key formula reference:

| Formula | Use |
|---------|-----|
| $\text{Gini} = 2 \times AUC - 1$ | Model discrimination |
| $\text{Factor} = PDO / \ln(2)$ | Scorecard scaling |
| $\text{Score}_i = -\beta_i \times \text{Factor}$ | Score per dummy |
| $PD = 1 / (1 + e^{(\text{Score}-\text{Offset})/\text{Factor}})$ | Score → PD |

**Next → L03:** Build the LGD (two-stage) and EAD models, then compute Expected Loss = PD × LGD × EAD at the portfolio level.
